# JOINs en PySpark — Daniel Guzmán

**Semana:** 02  
**Actividad:** 02 — JOINs en PySpark  
**Dataset:** Financial Transactions  
**Objetivo:** Conectar las tablas del modelo financiero usando JOINs en PySpark y analizar fraude.

## Modelo de datos — Financial Transactions

Antes de hacer JOINs es importante entender cómo se relacionan las tablas del dataset.

| Tabla | Llave primaria | Llave foránea hacia |
|-------|----------------|---------------------|
| transactions_data | id | client_id → users_data.id, card_id → cards_data.id, mcc → mcc_codes.mcc, id → train_fraud_labels.id |
| users_data | id | — |
| cards_data | id | client_id → users_data.id |
| mcc_codes | mcc | — |
| train_fraud_labels | id | id → transactions_data.id |

### Diagrama de relaciones

transactions_data es la tabla principal.

users_data.id
→ transactions_data.client_id

cards_data.id
→ transactions_data.card_id

mcc_codes.mcc
→ transactions_data.mcc

train_fraud_labels.id
→ transactions_data.id

cards_data.client_id
→ users_data.id

In [0]:
MI_NOMBRE = "daniel"

VOL = f"/Volumes/workspace/default/week_2_{MI_NOMBRE}"

display(dbutils.fs.ls(VOL))

In [0]:
MI_NOMBRE = "daniel"

VOL = f"/Volumes/workspace/default/week_2_{MI_NOMBRE}"

display(dbutils.fs.ls(VOL))

In [0]:
from pyspark.sql import functions as F

MI_NOMBRE = "daniel"
VOL = f"/Volumes/workspace/default/week_2_{MI_NOMBRE}"

# Tablas CSV
df_transactions = (
    spark.read.format("csv")
    .option("header", "true")
    .option("inferSchema", "true")
    .load(f"{VOL}/transactions_data.csv")
)

df_users = (
    spark.read.format("csv")
    .option("header", "true")
    .option("inferSchema", "true")
    .load(f"{VOL}/users_data.csv")
)

df_cards = (
    spark.read.format("csv")
    .option("header", "true")
    .option("inferSchema", "true")
    .load(f"{VOL}/cards_data.csv")
)

# MCC codes JSON
df_mcc_raw = (
    spark.read
    .option("multiLine", "true")
    .json(f"{VOL}/mcc_codes.json")
)

print("Tablas CSV y JSON MCC cargadas correctamente")

In [0]:
# Pivotar mcc_codes: de ancho a largo
mcc_cols = df_mcc_raw.columns

stack_expr = (
    f"stack({len(mcc_cols)}, "
    + ", ".join([f"'{c}', `{c}`" for c in mcc_cols])
    + ") as (mcc_str, description)"
)

df_mcc = (
    df_mcc_raw
    .select(F.expr(stack_expr))
    .withColumn("mcc", F.col("mcc_str").cast("int"))
    .drop("mcc_str")
)

print(f"Categorías MCC: {df_mcc.count():,}")
df_mcc.show(5, truncate=False)

In [0]:
try:
    df_fraud = spark.read.parquet(f"{VOL}/train_fraud_labels.parquet")
    print("✓ Parquet cargado correctamente")
except Exception:
    print("⚠ Parquet no encontrado — convirtiendo desde JSON...")
    df_fraud = spark.read.option("multiLine", "true").json(f"{VOL}/train_fraud_labels.json")
    df_fraud.write.mode("overwrite").parquet(f"{VOL}/train_fraud_labels.parquet")
    df_fraud = spark.read.parquet(f"{VOL}/train_fraud_labels.parquet")
    print("✓ Conversión completada y Parquet relanzado")

In [0]:
for nombre, dataframe in [
    ("transactions", df_transactions),
    ("users", df_users),
    ("cards", df_cards),
    ("mcc", df_mcc),
    ("fraud", df_fraud)
]:
    print(f"{nombre}: {dataframe.count():,} registros | {len(dataframe.columns)} columnas")

## Carga de tablas y formatos

En esta parte se cargaron las 5 tablas del modelo financiero:

- `transactions_data.csv`: tabla principal de transacciones.
- `users_data.csv`: información de clientes.
- `cards_data.csv`: información de tarjetas.
- `mcc_codes.json`: códigos de categoría de comercio.
- `train_fraud_labels.parquet`: etiquetas de fraude.

### ¿Por qué JSON se lee diferente al CSV?

CSV es un formato tabular: cada fila representa un registro y las columnas están separadas por delimitadores.

JSON puede tener estructuras más flexibles, anidadas o con múltiples niveles. En este caso, `mcc_codes.json` viene como un objeto donde cada clave es un código MCC y cada valor es la descripción. Por eso Spark lo leyó inicialmente como una fila con muchas columnas, y fue necesario convertirlo a formato largo con columnas `mcc` y `description`.

### ¿Qué hace multiLine?

La opción `multiLine=True` permite leer archivos JSON que están escritos en varias líneas. Sin esa opción, Spark puede interpretar cada línea como un JSON independiente y generar errores o una estructura incorrecta.

### ¿Qué ventajas tiene Parquet sobre JSON?

Parquet es un formato columnar, comprimido y optimizado para análisis. En Spark suele ser más eficiente que JSON porque lee solo las columnas necesarias, conserva tipos de datos y ocupa menos espacio. JSON es más flexible y legible, pero normalmente es menos eficiente para procesamiento analítico grande.